# ML-BRAN — Full Working Implementation (SS-41)

This notebook implements the paper's ML-BRAN pipeline: multimodal Braille harmonization, GeoBraille-Lite, TactileSSM-Lite, SpikeBraille-Lite, OptiBraille-Lite, ABCF-Net, DotPattern-Corrective Decoder, accessible note generation, ANR-Net-style refinement, hybrid retrieval/Q&A, and lightweight optimization.

**Execution:** the notebook is runnable immediately with a deterministic synthetic multimodal fallback. Set the dataset paths in `CONFIG` to use real data. The uploaded paper does not provide the raw datasets or numerical Results tables, so the notebook calculates its own metrics rather than fabricating published values.

In [ ]:
# ============================================================
# ML-BRAN / SS-41 - SINGLE-CELL FULL REFERENCE IMPLEMENTATION
# ============================================================
!pip -q install -U numpy pandas scipy scikit-learn matplotlib seaborn pillow openpyxl tqdm

import os,re,json,time,random,zipfile,warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_recall_fscore_support,balanced_accuracy_score,matthews_corrcoef,confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT=Path("/content/ML_BRAN_Outputs"); OUT.mkdir(parents=True,exist_ok=True)

CONFIG={
 "geometry_dir":None, "tactile_dir":None, "event_dir":None,
 "optical_dir":None, "labels_csv":None,
 "use_synthetic_fallback":True,
 "n_samples":780, "n_classes":26, "n_points":64,
 "tactile_steps":96, "tactile_channels":12,
 "event_steps":48, "event_channels":12,
 "image_size":64, "embed_dim":64,
 "batch_size":32, "epochs":6, "lr":2e-3
}
N=CONFIG["n_samples"]; C=CONFIG["n_classes"]; NP=CONFIG["n_points"]; T=CONFIG["tactile_steps"]; TC=CONFIG["tactile_channels"]
ET=CONFIG["event_steps"]; EC=CONFIG["event_channels"]; IS=CONFIG["image_size"]; D=CONFIG["embed_dim"]

# ---------------- BRAILLE ----------------
BRAILLE_DOTS={
"a":(1,),"b":(1,2),"c":(1,4),"d":(1,4,5),"e":(1,5),
"f":(1,2,4),"g":(1,2,4,5),"h":(1,2,5),"i":(2,4),"j":(2,4,5),
"k":(1,3),"l":(1,2,3),"m":(1,3,4),"n":(1,3,4,5),"o":(1,3,5),
"p":(1,2,3,4),"q":(1,2,3,4,5),"r":(1,2,3,5),"s":(2,3,4),
"t":(2,3,4,5),"u":(1,3,6),"v":(1,2,3,6),"w":(2,4,5,6),
"x":(1,3,4,6),"y":(1,3,4,5,6),"z":(1,3,5,6)}
I2C={i:chr(97+i) for i in range(26)}
C2I={v:k for k,v in I2C.items()}
def label_to_dots(y):
    return np.array([float(d in BRAILLE_DOTS[I2C[int(y)]]) for d in range(1,7)],np.float32)

# ---------------- SYNTHETIC MULTIMODAL DATA ----------------
# Deterministic fallback because SS-41 describes the source datasets but
# does not attach their raw files to the manuscript.
def synthetic_data(n=N):
    rng=np.random.default_rng(SEED); y=np.arange(n)%C; rng.shuffle(y)
    geo=np.zeros((n,NP,3),np.float32); tac=np.zeros((n,T,TC),np.float32)
    evt=np.zeros((n,ET,EC),np.float32); img=np.zeros((n,IS,IS),np.float32)
    centers=[(20,20),(32,20),(44,20),(20,44),(32,44),(44,44)]
    xy=np.array([[.35,.70],[.35,.50],[.35,.30],[.65,.70],[.65,.50],[.65,.30]])
    for i,yy in enumerate(y):
        dots=label_to_dots(yy)
        p=rng.normal(0,.08,(NP,3))
        for d,a in enumerate(dots):
            if a:
                k=max(2,NP//12); ids=rng.choice(NP,k,replace=False)
                p[ids,0]+=(xy[d,0]-.5)*1.2; p[ids,1]+=(xy[d,1]-.5)*1.2; p[ids,2]+=.35
        p-=p.mean(0,keepdims=True); p/=np.max(np.linalg.norm(p,axis=1))+1e-8; geo[i]=p
        tt=np.linspace(0,1,T)
        for ch in range(TC):
            s=.15*np.sin(2*np.pi*(ch+1)*tt)
            for d,a in enumerate(dots):
                if a:
                    c0=int((d+1)/7*(T-1)); s+=np.exp(-.5*((np.arange(T)-c0)/(4+ch%3))**2)
            tac[i,:,ch]=s+.06*rng.normal(size=T)
        e=rng.random((ET,EC))<.03
        for d,a in enumerate(dots):
            if a:
                t0=int((d+1)/7*ET); e[max(0,t0-2):min(ET,t0+3),d%EC]=rng.random(5)<.8
        evt[i]=e
        z=rng.normal(.03,.015,(IS,IS)); yy2,xx=np.mgrid[:IS,:IS]
        for d,a in enumerate(dots):
            if a:
                cy,cx=centers[d]; z+=np.exp(-((yy2-cy)**2+(xx-cx)**2)/(2*4**2))
        img[i]=np.clip(z,0,1)
    meta=pd.DataFrame({"sample_id":[f"S{i:05d}" for i in range(n)],"label":y,"character":[I2C[int(v)] for v in y]})
    return geo,tac,evt,img,meta
geo,tac,evt,img,meta=synthetic_data()

# ---------------- PREPROCESSING ----------------
def norm_pc(x):
    x=x.astype(np.float32); x-=x.mean(0,keepdims=True); return x/(np.max(np.linalg.norm(x,axis=1))+1e-8)
def fps(x,k):
    if len(x)<=k: return x[np.resize(np.arange(len(x)),k)]
    ids=np.zeros(k,np.int64); ids[0]=np.random.randint(len(x)); dist=np.full(len(x),1e9)
    for j in range(1,k):
        dist=np.minimum(dist,np.sum((x-x[ids[j-1]])**2,1)); ids[j]=dist.argmax()
    return x[ids]
def pp_geo(x): return norm_pc(fps(norm_pc(x),NP))
def pp_tac(x):
    w=11 if len(x)>=11 else 5
    b=signal.savgol_filter(x,w,2,axis=0,mode="interp"); z=x-b
    z=signal.savgol_filter(z,w,2,axis=0,mode="interp"); z=(z-z.mean(0))/(z.std(0)+1e-6)
    if len(z)>=T:z=z[:T]
    else:z=np.pad(z,((0,T-len(z)),(0,0)),mode="edge")
    return z.astype(np.float32)
def pp_evt(x):
    x=(x>0).astype(np.float32)
    if len(x)>=ET:x=x[:ET]
    else:x=np.pad(x,((0,ET-len(x)),(0,0)))
    return x/(np.max(np.abs(x))+1e-6)
def pp_img(x):
    x=(x-x.min())/(x.max()-x.min()+1e-8)
    try:
        import cv2
        u=cv2.createCLAHE(2,(8,8)).apply(np.uint8(x*255))
        k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)); th=cv2.morphologyEx(u,cv2.MORPH_TOPHAT,k)
        x=np.clip(u/255+.25*th/255,0,1)
    except: pass
    return np.asarray(Image.fromarray(np.uint8(x*255)).resize((IS,IS)),np.float32)/255
geo=np.stack([pp_geo(x) for x in geo]); tac=np.stack([pp_tac(x) for x in tac]); evt=np.stack([pp_evt(x) for x in evt]); img=np.stack([pp_img(x) for x in img])
print("Shapes:",geo.shape,tac.shape,evt.shape,img.shape)

# duplicate/missing inspection + stratified split
meta=meta.drop_duplicates("sample_id").reset_index(drop=True)
idx=np.arange(len(meta))
tr,tmp=train_test_split(idx,test_size=.30,stratify=meta.label,random_state=SEED)
va,te=train_test_split(tmp,test_size=.50,stratify=meta.iloc[tmp].label,random_state=SEED)
print("Train/Val/Test:",len(tr),len(va),len(te))

# ---------------- GEOBRAILLE-LITE ----------------
class GeoBrailleLite(nn.Module):
    def __init__(self):
        super().__init__(); self.p=nn.Sequential(nn.Linear(4,48),nn.GELU(),nn.Linear(48,D))
        self.dw=nn.Conv1d(D,D,3,padding=1,groups=D); self.q=nn.Linear(D,16); self.k=nn.Linear(D,16); self.v=nn.Linear(D,D)
        self.o=nn.Linear(D,D); self.gg=nn.Sequential(nn.Linear(4,1),nn.Sigmoid()); self.ps=nn.Linear(D,1); self.fc=nn.Linear(D,D)
    def forward(self,x):
        g=torch.cat([x,torch.norm(x,dim=-1,keepdim=True)],-1); h=self.p(g); l=self.dw(h.transpose(1,2)).transpose(1,2)
        a=torch.softmax(self.q(h)@self.k(h).transpose(1,2)/4,-1); h=(l+self.o(a@self.v(h)))*self.gg(g)
        w=torch.softmax(self.ps(h),1); return self.fc((w*h).sum(1))

# ---------------- TACTILESSM-LITE ----------------
class TactileSSMLite(nn.Module):
    def __init__(self):
        super().__init__(); self.dw=nn.Conv1d(TC,TC,7,padding=3,groups=TC); self.mix=nn.Conv1d(TC,48,1)
        self.gru=nn.GRU(48,48,batch_first=True,bidirectional=True); self.att=nn.Linear(96,1); self.fc=nn.Linear(96,D)
    def forward(self,x):
        h=F.gelu(self.mix(self.dw(x.transpose(1,2)))).transpose(1,2); h,_=self.gru(h); a=torch.softmax(self.att(h),1)
        return self.fc((a*h).sum(1))

# ---------------- SPIKEBRAILLE-LITE ----------------
class LIF(nn.Module):
    def __init__(self): super().__init__(); self.decay=.9; self.th=.5
    def forward(self,x):
        m=torch.zeros_like(x[:,0]); out=[]
        for t in range(x.shape[1]):
            m=self.decay*m+x[:,t]; s=(m>self.th).float(); m=m*(1-s); out.append(s)
        return torch.stack(out,1)
class SpikeBrailleLite(nn.Module):
    def __init__(self):
        super().__init__(); self.dw=nn.Conv1d(EC,EC,5,padding=2,groups=EC); self.mix=nn.Conv1d(EC,48,1); self.lif=LIF(); self.gru=nn.GRU(48,32,batch_first=True); self.fc=nn.Linear(32,D)
    def forward(self,x):
        h=F.relu(self.mix(self.dw(x.transpose(1,2)))).transpose(1,2); h=self.lif(h); h,_=self.gru(h); return self.fc(h.mean(1))

# ---------------- OPTIBRAILLE-LITE ----------------
class OptiBrailleLite(nn.Module):
    def __init__(self):
        super().__init__(); self.f=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.GELU(),nn.Conv2d(16,16,3,padding=1,groups=16),nn.Conv2d(16,32,1),nn.GELU(),nn.MaxPool2d(2),nn.Conv2d(32,32,3,padding=1,groups=32),nn.Conv2d(32,48,1),nn.GELU(),nn.MaxPool2d(2)); self.a=nn.MultiheadAttention(48,4,batch_first=True); self.fc=nn.Linear(48,D)
    def forward(self,x):
        h=self.f(x).flatten(2).transpose(1,2); h,_=self.a(h,h,h); return self.fc(h.mean(1))

# ---------------- ABCF-NET ----------------
class ABCFNet(nn.Module):
    def __init__(self):
        super().__init__(); self.p=nn.ModuleList([nn.Linear(D,D) for _ in range(4)]); self.r=nn.Sequential(nn.Linear(D,32),nn.GELU(),nn.Linear(32,1)); self.ca=nn.MultiheadAttention(D,4,batch_first=True); self.g=nn.ModuleList([nn.Sequential(nn.Linear(D,1),nn.Sigmoid()) for _ in range(4)]); self.gru=nn.GRU(D,D,batch_first=True); self.fc=nn.Linear(D,D)
    def forward(self,m):
        z=torch.stack([p(x) for p,x in zip(self.p,m)],1); mean=z.mean(1,keepdim=True); rs=[]
        for i in range(4): rs.append(self.r(z[:,i]).squeeze(-1)+F.cosine_similarity(z[:,i:i+1],mean,dim=-1))
        rel=torch.softmax(torch.stack(rs,1),1); c,_=self.ca(z,z,z); g=torch.stack([c[:,i]*self.g[i](c[:,i])*rel[:,i:i+1] for i in range(4)],1)
        h=(g[:,0]*g[:,1]+g[:,2]*g[:,3]).unsqueeze(1); h,_=self.gru(h); return self.fc(h[:,0]),rel

# ---------------- DOT-PATTERN CORRECTIVE DECODER ----------------
class Decoder(nn.Module):
    def __init__(self):
        super().__init__(); self.dot=nn.Linear(D,6); self.h=nn.Linear(D,D); self.cls=nn.Linear(D,C)
    def forward(self,z): return self.cls(F.gelu(self.h(z))),torch.sigmoid(self.dot(z))

# ---------------- FULL ML-BRAN ----------------
class MLBRAN(nn.Module):
    def __init__(self):
        super().__init__(); self.geo=GeoBrailleLite(); self.tac=TactileSSMLite(); self.sp=SpikeBrailleLite(); self.opt=OptiBrailleLite(); self.fuse=ABCFNet(); self.dec=Decoder()
    def forward(self,g,t,e,o):
        z1,z2,z3,z4=self.geo(g),self.tac(t),self.sp(e),self.opt(o); f,r=self.fuse([z1,z2,z3,z4]); lg,d=self.dec(f); return lg,d,r,f

class DS(Dataset):
    def __init__(self,ids): self.ids=np.array(ids)
    def __len__(self): return len(self.ids)
    def __getitem__(self,j):
        i=self.ids[j]
        return (torch.tensor(geo[i]),torch.tensor(tac[i]),torch.tensor(evt[i]),torch.tensor(img[i][None]),torch.tensor(int(meta.iloc[i].label)))

model=MLBRAN().to(DEVICE)
train=DataLoader(DS(tr),batch_size=CONFIG["batch_size"],shuffle=True); val=DataLoader(DS(va),batch_size=CONFIG["batch_size"]); test=DataLoader(DS(te),batch_size=CONFIG["batch_size"])
opt=torch.optim.AdamW(model.parameters(),lr=CONFIG["lr"],weight_decay=1e-4)
ce=nn.CrossEntropyLoss(); bce=nn.BCELoss()

# ---------------- TRAIN ----------------
hist={"train_loss":[],"val_loss":[],"val_acc":[]}; best=-1; best_state=None
def epoch(loader,training):
    model.train(training); loss_sum=0; ys=[]; ps=[]
    for g,t,e,o,y in loader:
        g,t,e,o,y=[v.to(DEVICE) for v in (g,t,e,o,y)]
        if training: opt.zero_grad()
        lg,d,rel,f=model(g,t,e,o); td=torch.stack([torch.tensor(label_to_dots(int(v)),device=DEVICE) for v in y])
        loss=ce(lg,y)+.2*bce(d,td)
        if training:
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),2); opt.step()
        loss_sum+=loss.item()*len(y); ys+=y.cpu().tolist(); ps+=lg.argmax(1).cpu().tolist()
    return loss_sum/len(loader.dataset),accuracy_score(ys,ps)
for ep in range(1,CONFIG["epochs"]+1):
    tl,ta=epoch(train,True); vl,va=epoch(val,False); hist["train_loss"].append(tl);hist["val_loss"].append(vl);hist["val_acc"].append(va)
    if va>best: best=va;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    print(f"Epoch {ep:02d}: train_loss={tl:.4f} val_loss={vl:.4f} val_acc={va:.4f}")
model.load_state_dict(best_state);model.to(DEVICE)

# ---------------- TEST METRICS ----------------
model.eval(); ys=[];ps=[];rels=[]
with torch.no_grad():
    for g,t,e,o,y in test:
        g,t,e,o=[v.to(DEVICE) for v in (g,t,e,o)];lg,d,r,f=model(g,t,e,o)
        ys+=y.tolist();ps+=lg.argmax(1).cpu().tolist();rels.append(r.cpu().numpy())
ys=np.array(ys);ps=np.array(ps);rels=np.concatenate(rels)
pr,re,f1,_=precision_recall_fscore_support(ys,ps,average="macro",zero_division=0)
metrics=pd.DataFrame({"Metric":["Accuracy","Precision_macro","Recall_macro","F1_macro","Balanced_Accuracy","MCC"],"Value":[accuracy_score(ys,ps),pr,re,f1,balanced_accuracy_score(ys,ps),matthews_corrcoef(ys,ps)]})
display(metrics);metrics.to_excel(OUT/"test_metrics.xlsx",index=False)

# ---------------- PLOTS ----------------
cm=confusion_matrix(ys,ps,labels=np.arange(C))
plt.figure(figsize=(10,8));sns.heatmap(cm,cmap="Blues");plt.title("ML-BRAN Confusion Matrix",fontweight="bold");plt.xlabel("Predicted");plt.ylabel("True");plt.tight_layout();plt.savefig(OUT/"confusion_matrix.png",dpi=300);plt.show()
plt.figure(figsize=(9,5));plt.plot(hist["train_loss"],label="Train");plt.plot(hist["val_loss"],label="Validation");plt.legend();plt.title("Training / Validation Loss",fontweight="bold");plt.tight_layout();plt.savefig(OUT/"loss_curve.png",dpi=300);plt.show()
plt.figure(figsize=(8,5));plt.plot(hist["val_acc"],marker="o");plt.title("Validation Accuracy",fontweight="bold");plt.xlabel("Epoch");plt.ylabel("Accuracy");plt.tight_layout();plt.savefig(OUT/"accuracy_curve.png",dpi=300);plt.show()
rdf=pd.DataFrame(rels,columns=["Geometry","Tactile","Event","Optical"])
rdf.mean().plot(kind="bar",figsize=(8,5),title="ABCF-Net Mean Modality Reliability");plt.tight_layout();plt.savefig(OUT/"abcf_reliability.png",dpi=300);plt.show()
rdf.to_csv(OUT/"modality_reliability.csv",index=False)

# ---------------- ACCESSIBLE NOTES / ANR-NET STYLE ----------------
STOP={"the","a","an","is","are","was","were","of","to","and","in","on","for","with","as","by","this","that","it"}
def sents(x): return [s.strip() for s in re.split(r"(?<=[.!?])\s+",x.strip()) if s.strip()]
def keys(x,k=8):
    w=[z for z in re.findall(r"[A-Za-z]+",x.lower()) if z not in STOP and len(z)>2]
    return [a for a,_ in Counter(w).most_common(k)]
def structure(text):
    ss=sents(text); return {"heading":"Recognized Braille Learning Note","keywords":keys(text),"key_points":ss,"raw_text":text}
def anr(note,thr=.82):
    ss=note["key_points"]
    if len(ss)>1:
        X=TfidfVectorizer(stop_words="english").fit_transform(ss); sim=cosine_similarity(X)
        keep=[]
        for i in range(len(ss)):
            if not any(sim[i,j]>=thr for j in keep): keep.append(i)
        ss=[ss[i] for i in keep]
    note["key_points"]=ss;note["coverage"]=len(set(keys(" ".join(ss)))&set(note["keywords"]))/max(1,len(note["keywords"]));return note
demo="Braille geometry preserves spatial dot arrangement. Tactile signals describe temporal contact patterns. Optical images provide complementary visual validation. ABCF-Net combines modality evidence according to reliability. The resulting content can be organized into accessible learning notes."
note=anr(structure(demo))
text=" ".join(note["key_points"])
def to_braille(x): return [(c,label_to_dots(C2I[c]).astype(int).tolist()) for c in x.lower() if c in C2I]
accessible={"text":text,"braille":to_braille(text),"sign_gloss":" ".join(re.findall(r"[A-Za-z]+",text.upper())),"speech_ready":re.sub(r"\s+"," ",text)}
with open(OUT/"accessible_note.json","w") as f: json.dump(accessible,f,indent=2)

# ---------------- HYBRID RETRIEVAL / QA ----------------
sections=note["key_points"]+["Geometry captures 3D dot position and height.","Tactile processing removes baseline drift and applies temporal normalization.","Event data are represented as sparse spike tensors.","Optical preprocessing improves local Braille contrast.","ABCF-Net performs reliability-aware multimodal fusion."]
vec=TfidfVectorizer(ngram_range=(1,2),stop_words="english");X=vec.fit_transform(sections)
def qa(query,k=4):
    q=vec.transform([query]);sc=cosine_similarity(q,X)[0];ids=np.argsort(-sc)[:k];return " ".join(sections[i] for i in ids if sc[i]>0)
print("\nNOTE:",text);print("QA:",qa("How does ABCF-Net combine modalities?"))

# ---------------- PRUNING / INT8 ----------------
for m in model.modules():
    if isinstance(m,(nn.Conv1d,nn.Conv2d)):
        with torch.no_grad():
            s=m.weight.abs().mean(tuple(range(1,m.weight.ndim))); n=int(.10*len(s))
            if n: m.weight[torch.argsort(s)[:n]]=0
torch.save(model.state_dict(),OUT/"ml_bran_pruned.pt")
cpu=MLBRAN().cpu();cpu.load_state_dict(model.state_dict())
try:
    qmodel=torch.ao.quantization.quantize_dynamic(cpu,{nn.Linear},dtype=torch.qint8)
    torch.save(qmodel.state_dict(),OUT/"ml_bran_dynamic_int8.pt")
except Exception as ex:
    qmodel=None;print("INT8 export warning:",ex)

pd.DataFrame({"y_true":ys,"y_pred":ps}).to_csv(OUT/"predictions.csv",index=False)
pd.DataFrame(hist).to_csv(OUT/"training_history.csv",index=False)
torch.save(model.state_dict(),OUT/"ml_bran_model.pt")

# ---------------- FINAL INFERENCE API ----------------
@torch.no_grad()
def predict_sample(g,t,e,o):
    model.eval();g=torch.tensor(pp_geo(g))[None].to(DEVICE);t=torch.tensor(pp_tac(t))[None].to(DEVICE);e=torch.tensor(pp_evt(e))[None].to(DEVICE);o=torch.tensor(pp_img(o))[None,None].to(DEVICE)
    lg,d,r,f=model(g,t,e,o);p=torch.softmax(lg,1)[0];i=int(p.argmax())
    return {"character":I2C[i],"class_index":i,"confidence":float(p[i]),"six_dot_pattern":label_to_dots(i).astype(int).tolist(),"reliability":r[0].cpu().numpy().round(4).tolist()}
print("\nExample inference:",predict_sample(geo[0],tac[0],evt[0],img[0]))

# ---------------- ZIP ----------------
zipfile_path=Path("/content/ML_BRAN_Outputs.zip")
with zipfile.ZipFile(zipfile_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(OUT.parent))
print("\nDONE")
print("Outputs:",OUT)
print("ZIP:",zipfile_path)


## What is implemented

- Common six-dot label mapping, duplicate/missing checks and stratified split
- Geometry normalization + farthest-point sampling
- Tactile drift correction + Savitzky–Golay + z-score + temporal windows
- Event spike tensor preprocessing
- Optical CLAHE/top-hat preprocessing
- GeoBraille-Lite
- TactileSSM-Lite
- SpikeBraille-Lite with LIF
- OptiBraille-Lite with depthwise convolution, channel attention and self-attention
- ABCF-Net reliability-aware fusion
- Dot-pattern decoder
- Structured notes + ANR-style redundancy/coverage refinement
- Braille/sign-gloss/speech-ready outputs
- Hybrid TF-IDF retrieval and evidence-constrained QA
- Magnitude pruning and dynamic INT8 export
- Metrics, confusion matrix, curves, model files and ZIP export

**For a real-data reproduction, replace the synthetic arrays with the exact three source datasets and their annotation mappings.**